[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/personal-health-agent/binf4070-2026/blob/main/week-03/lab-03.ipynb)

# Lab 3: Fitbit Data, Visualizations, and an AI Summary
**Course:** BINF 4070 — The Future of Personal Health Assistant

**Week:** 3


We will authorize Fitbit API access, retrieve one day of data, make three
charts, and check a short summary against its evidence. Week 1 introduced
Colab, pandas, and matplotlib; Week 2 practiced interpreting clinical records.
Today we use those tools to examine wearable measurements.

Orson will demonstrate with his own Fitbit data. Your new Fitbit Air may have
no history yet, so **start with the synthetic data in class**. Live Fitbit
data is optional for the class exercises. In Part 3, use the
OpenAI API key already issued to you to generate a summary. Live Fitbit data
is required for homework if you received a Fitbit: use your own authorized
records from a day you wore and synced it. Students without a Fitbit use the
provided sample path and complete the same analyses. Keep credentials and
identifiable raw records out of submissions.

Run the setup cell, then the provided cells in order. Cells labeled
**Provided** need no edits. Complete the visible TODOs. During class, skip the
homework section and finish the two reflection questions at the end.

In [ ]:
#@title Provided - Part 0 - Install and import libraries
# Install every external library required by this notebook inside Colab.
# Major-version bounds keep the lab reproducible without freezing old patches.
%pip install -q "requests>=2.32,<3" "openai>=2,<3" "matplotlib>=3.8,<4" "pandas>=2.2,<3" "tiktoken>=0.9,<1"

import copy
import json
import os
import time
from datetime import datetime, timedelta, timezone
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
import requests
import tiktoken
from IPython.display import Markdown, display
from openai import OpenAI

print("Setup complete. All required libraries are installed and imported.")

## Part 0 — Fitbit API setup

The preceding slides covered pairing Fitbit Air with Google Health. Pairing
lets the device sync to your account; **OAuth authorization** separately lets
the course application read the record types you approve. The Fitbit API
requests below use Google Health API v4.

If you choose the synthetic path, run the readiness check and continue to
Part 1. If you choose live access:

1. Open the [course authorization helper](https://binf4070-health-oauth-helper-7pw5f3iaiq-ue.a.run.app/authorize).
   Use the **same Google Account** as Google Health.

2. Read the disclosure and linked privacy terms. To use live data in this
   lab, grant **all three read permissions**: activity for steps, health
   metrics for heart rate, and sleep. We use all three record types together.
3. Save the two values from the helper in Colab **Secrets** (the key icon):
   `GOOGLE_HEALTH_REFRESH_TOKEN` and `GOOGLE_HEALTH_HELPER_TOKEN`. Enable
   notebook access for both. They are a matching credential pair.
4. Run the check below. It reports availability only; it never prints values.

**If setup stalls, continue with synthetic data.** New devices may have no
steps or sleep history. Finish setup with the course team through a private
course channel.

Never paste credentials into notebook cells, outputs, submissions, messages,
or AI tools. The client exchanges them for a short-lived access token held in
memory. If either credential may have been exposed, revoke the course app in
[Google Account connections](https://myaccount.google.com/connections), delete
both Secrets, then authorize again. The course application requests no
medical-record scopes.

Reference: [Google Health authorization setup](https://developers.google.com/health/setup).

In [ ]:
#@title Provided - Part 0 - Check API credentials privately
# Safe credential check: report availability, never credential values.
def _colab_secret_is_available(name):
    if os.environ.get(name):
        return True
    try:
        from google.colab import userdata
        return bool(userdata.get(name))
    except Exception:
        # Outside Colab, a missing Secret, or notebook access not enabled.
        return False


secret_status = {
    name: _colab_secret_is_available(name)
    for name in (
        "GOOGLE_HEALTH_REFRESH_TOKEN",
        "GOOGLE_HEALTH_HELPER_TOKEN",
    )
}
for name, available in secret_status.items():
    print(f"{name}: {'available' if available else 'not available'}")

if all(secret_status.values()):
    print("Live Google Health access is ready to test; no credential was printed.")
else:
    print(
        "Live access is not ready yet. Add both Colab Secrets and enable "
        "notebook access, or continue with the equally valid synthetic path."
    )

### Provided - Part 0 — Data and API client

Run the next two cells; **no editing is needed**. The first creates small
fictional responses for September 18–24, 2026. The second provides
`list_health_data(type, day, force_sample=True)`:

- `True` reads the in-memory sample and makes no network request.
- `False` requests your authorized Google Health records for that local date.

Each sample day has 144 ten-minute Fitbit step intervals and heart-rate point
samples every ten minutes (144 points). September 22 deliberately has no
heart-rate observations.
The rest and movement patterns are designed teaching data, not real measurements.

The client handles token renewal and pagination. A live request error stops
the cell; it never silently replaces your data with samples. To change paths,
change the control yourself and rerun. Credentials are never printed or
included in an API URL.

In [ ]:
#@title Provided - Part 0 - Synthetic API responses
DATA_SOURCE = {
    "recordingMethod": "PASSIVELY_MEASURED",
    "application": {"packageName": "com.google.fitbit"},
    "device": {
        "manufacturer": "Google",
        "displayName": "Course Sample Tracker",
    },
    "platform": "FITBIT",
}


def _zulu(dt):
    return dt.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def _civil(dt):
    return {
        "date": {"year": dt.year, "month": dt.month, "day": dt.day},
        "time": {"hours": dt.hour, "minutes": dt.minute},
    }


def _interval(day, minute_of_day, duration_minutes=10):
    start = datetime.fromisoformat(day).replace(tzinfo=timezone.utc)
    start += timedelta(minutes=minute_of_day)
    end = start + timedelta(minutes=duration_minutes)
    return {
        "startTime": _zulu(start),
        "startUtcOffset": "0s",
        "endTime": _zulu(end),
        "endUtcOffset": "0s",
        "civilStartTime": _civil(start),
        "civilEndTime": _civil(end),
    }


def _ten_minute_step_counts(total, day_index):
    # Fictional morning, commute, lunch, and evening bouts, shifted by day.
    # Integer allocation preserves the specified daily total exactly.
    shift = [-1, 1, 0, 2, -2, 1, 0][day_index]
    bouts = [(44, 2, 80), (50, 3, 300), (75, 4, 190),
             (91, 2, 70), (105, 3, 280), (113, 4, 360)]
    weights = []
    for slot in range(144):
        weight = 0
        if 42 <= slot < 135:
            # Light incidental movement, including some observed zero bins.
            weight = (slot * 7 + day_index * 3) % 13 if slot % 4 else 0
            for center, half_width, height in bouts:
                distance = abs(slot - (center + shift))
                weight += max(0, half_width - distance) * (
                    height + day_index * (center % 5)
                )
        weights.append(weight)
    weight_sum = sum(weights)
    counts = [total * weight // weight_sum for weight in weights]
    remainder_order = sorted(
        range(144), key=lambda slot: total * weights[slot] % weight_sum,
        reverse=True,
    )
    for slot in remainder_order[:total - sum(counts)]:
        counts[slot] += 1
    return counts


def _ten_minute_heart_rates(step_counts, day_index):
    # Designed point samples: lower overnight values, daytime variation,
    # and brief recovery after the invented movement bouts. Not a health model.
    day_offset = [-1, -2, 2, -3, 0, 1, 0][day_index]
    variation = [-2, -1, 1, 2, 0, -1, 0]
    values = []
    for slot, count in enumerate(step_counts):
        baseline = 59 if slot < 42 or slot >= 138 else 70
        previous_count = step_counts[slot - 1] if slot else 0
        movement = min(48, count // 12 + previous_count // 30)
        values.append(
            baseline + day_offset + variation[(slot + day_index) % 7]
            + movement
        )
    return values


def _step_response(day, counts):
    # Keep one complete Fitbit-source stream, then add a small overlapping
    # fictional import stream so the source-selection exercise is visible.
    points = []
    for slot, count in enumerate(counts):
        steps = {"interval": _interval(day, slot * 10)}
        if count or slot % 2:
            steps["count"] = str(count)
        # Some returned true-zero points omit the default count field.
        points.append({
            "dataSource": copy.deepcopy(DATA_SOURCE),
            "steps": steps,
        })
    imported_source = copy.deepcopy(DATA_SOURCE)
    imported_source["application"]["packageName"] = (
        "edu.columbia.binf4070.synthetic.import"
    )
    imported_source["recordingMethod"] = "MANUAL"
    imported_source["device"]["displayName"] = "Course Sample Import"
    points.extend([
        {
            "dataSource": copy.deepcopy(imported_source),
            "steps": {
                "interval": _interval(day, 720),
                "count": "250",
            },
        },
        {
            "dataSource": copy.deepcopy(imported_source),
            "steps": {
                "interval": _interval(day, 840),
                "count": "0",
            },
        },
    ])
    points.sort(
        key=lambda point: point["steps"]["interval"]["startTime"],
        reverse=True,
    )  # v4 list responses are newest-first across all application sources.
    return {"dataPoints": points, "nextPageToken": ""}


def _heart_rate_response(day, values):
    points = []
    for slot, bpm in enumerate(values):
        observed = datetime.fromisoformat(day).replace(tzinfo=timezone.utc)
        observed += timedelta(minutes=slot * 10)
        points.append({
            "dataSource": copy.deepcopy(DATA_SOURCE),
            "heartRate": {
                "sampleTime": {
                    "physicalTime": _zulu(observed),
                    "utcOffset": "0s",
                    "civilTime": _civil(observed),
                },
                "metadata": {
                    "motionContext": "ACTIVE" if bpm >= 95 else "SEDENTARY",
                    "sensorLocation": "WRIST",
                },
                "beatsPerMinute": str(bpm),
            },
        })
    points.reverse()  # v4 list responses are newest-first.
    return {"dataPoints": points, "nextPageToken": ""}


def _sleep_response(day, minutes_asleep, minutes_awake):
    wake = datetime.fromisoformat(day).replace(tzinfo=timezone.utc)
    wake += timedelta(hours=7)
    start = wake - timedelta(minutes=minutes_asleep + minutes_awake)
    # Authored teaching sequence, not a reconstruction of a person's night.
    stage_totals = {
        "AWAKE": minutes_awake,
        "LIGHT": (minutes_asleep * 55 + 50) // 100,
        "DEEP": (minutes_asleep * 20 + 50) // 100,
    }
    stage_totals["REM"] = minutes_asleep - stage_totals["LIGHT"] - stage_totals["DEEP"]
    sequence = [
        "AWAKE", "LIGHT", "DEEP", "LIGHT", "REM", "AWAKE", "LIGHT",
        "DEEP", "LIGHT", "REM", "AWAKE", "LIGHT", "REM", "AWAKE",
    ]
    occurrences = {stage: sequence.count(stage) for stage in stage_totals}
    used = {stage: 0 for stage in stage_totals}
    stages = []
    cursor = start
    for stage in sequence:
        base, remainder = divmod(stage_totals[stage], occurrences[stage])
        duration = base + (used[stage] < remainder)
        used[stage] += 1
        if duration == 0:
            continue
        end = cursor + timedelta(minutes=duration)
        stages.append({
            "startTime": _zulu(cursor), "startUtcOffset": "0s",
            "endTime": _zulu(end), "endUtcOffset": "0s", "type": stage,
        })
        cursor = end
    stage_summary = [
        {
            "type": stage,
            "minutes": str(sum(
                (datetime.fromisoformat(item["endTime"].replace("Z", "+00:00"))
                 - datetime.fromisoformat(item["startTime"].replace("Z", "+00:00"))).total_seconds() / 60
                for item in stages if item["type"] == stage
            )).removesuffix(".0"),
            "count": str(sum(item["type"] == stage for item in stages)),
        }
        for stage in stage_totals if any(item["type"] == stage for item in stages)
    ]
    point = {
        "name": f"users/me/dataTypes/sleep/dataPoints/sample-{day}",
        "dataSource": copy.deepcopy(DATA_SOURCE),
        "sleep": {
            "interval": {
                "startTime": _zulu(start),
                "startUtcOffset": "0s",
                "endTime": _zulu(wake),
                "endUtcOffset": "0s",
                "civilStartTime": _civil(start),
                "civilEndTime": _civil(wake),
            },
            "type": "STAGES",
            "stages": stages,
            "metadata": {
                "processed": True,
                "nap": False,
                "manuallyEdited": False,
                "stagesStatus": "SUCCEEDED",
            },
            "summary": {
                "minutesInSleepPeriod": str(minutes_asleep + minutes_awake),
                "minutesAsleep": str(minutes_asleep),
                "minutesAwake": str(minutes_awake),
                "stagesSummary": stage_summary,
            },
        },
    }
    return {"dataPoints": [point], "nextPageToken": ""}


SAMPLE_DAYS = [
    (datetime(2026, 9, 18) + timedelta(days=i)).strftime("%Y-%m-%d")
    for i in range(7)
]
step_totals = [5210, 7840, 6435, 9020, 4880, 7350, 6815]
sleep_values = [
    (421, 38), (388, 52), (447, 31), (365, 61),
    (432, 35), (405, 44), (414, 40),
]

SAMPLE_RESPONSES = {}
for day_index, (day, total, sleep) in enumerate(zip(
    SAMPLE_DAYS, step_totals, sleep_values
)):
    counts = _ten_minute_step_counts(total, day_index)
    hrs = [] if day == "2026-09-22" else _ten_minute_heart_rates(counts, day_index)
    SAMPLE_RESPONSES[("steps", day)] = _step_response(day, counts)
    SAMPLE_RESPONSES[("heart-rate", day)] = _heart_rate_response(day, hrs)
    SAMPLE_RESPONSES[("sleep", day)] = _sleep_response(day, *sleep)

# A valid complete list response may omit the optional empty page token.
SAMPLE_RESPONSES[("sleep", SAMPLE_DAYS[1])].pop("nextPageToken")

def sample_list(data_type, day):
    key = (data_type, day)
    if key not in SAMPLE_RESPONSES:
        raise ValueError(
            f"No sample fixture for {data_type!r} on {day}. "
            f"Use {SAMPLE_DAYS[0]} through {SAMPLE_DAYS[-1]}."
        )
    return copy.deepcopy(SAMPLE_RESPONSES[key])


print(
    f"Loaded {len(SAMPLE_DAYS)} sample days: "
    f"{SAMPLE_DAYS[0]} through {SAMPLE_DAYS[-1]}."
)

In [ ]:
#@title Provided - Part 0 - Google Health API client
GOOGLE_HEALTH_BASE = "https://health.googleapis.com/v4"
COURSE_HELPER_BASE_URL = "https://binf4070-health-oauth-helper-7pw5f3iaiq-ue.a.run.app"
_ACCESS_TOKEN_CACHE = {
    "value": None,
    "expires_at": 0.0,
}

def _google_health_secret(name):
    value = os.environ.get(name)
    if value:
        return value
    from google.colab import userdata
    return userdata.get(name)


def _access_token():
    now = time.monotonic()
    if (
        _ACCESS_TOKEN_CACHE["value"]
        and now < _ACCESS_TOKEN_CACHE["expires_at"]
    ):
        return _ACCESS_TOKEN_CACHE["value"]

    refresh_token = _google_health_secret("GOOGLE_HEALTH_REFRESH_TOKEN")
    helper_token = _google_health_secret("GOOGLE_HEALTH_HELPER_TOKEN")

    response = requests.post(
        f"{COURSE_HELPER_BASE_URL}/refresh",
        json={"refresh_token": refresh_token},
        headers={
            "Authorization": f"Bearer {helper_token}",
            "Accept": "application/json",
        },
        timeout=30,
        allow_redirects=False,
    )
    response.raise_for_status()
    payload = response.json()
    access_token = payload["access_token"]
    expires_in = payload["expires_in"]

    # Keep the access token only in runtime memory and refresh at least one
    # minute before the helper-reported expiry.
    _ACCESS_TOKEN_CACHE.update(
        value=access_token,
        expires_at=time.monotonic() + expires_in - 60,
    )
    return access_token


def list_health_data(data_type, day, force_sample=True):
    '''Return one date of typed DataPoints.

    The synthetic path is guaranteed. Private live mode is for a Fitbit-path
    student who authorized the class OAuth client for their account.
    '''
    filter_fields = {
        "steps": "steps.interval.civil_start_time",
        "heart-rate": "heart_rate.sample_time.civil_time",
        "sleep": "sleep.interval.civil_end_time",
    }
    filter_field = filter_fields[data_type]
    next_day = (
        datetime.strptime(day, "%Y-%m-%d") + timedelta(days=1)
    ).strftime("%Y-%m-%d")

    if force_sample:
        return sample_list(data_type, day)

    token = _access_token()
    base_params = {
        # Google's maximum for sleep is 25; 1000 is a conservative size for
        # the other two data types (whose documented maximum is 10000).
        "pageSize": 25 if data_type == "sleep" else 1000,
        "filter": (
            f'{filter_field} >= "{day}" AND '
            f'{filter_field} < "{next_day}"'
        ),
    }
    endpoint = (
        f"{GOOGLE_HEALTH_BASE}/users/me/dataTypes/{data_type}/dataPoints"
    )

    all_points = []
    page_token = None
    seen_page_tokens = set()

    while True:
        params = dict(base_params)
        if page_token:
            params["pageToken"] = page_token

        response = requests.get(
            endpoint,
            headers={
                "Authorization": f"Bearer {token}",
                "Accept": "application/json",
            },
            params=params,
            timeout=30,
        )
        response.raise_for_status()
        page = response.json()
        all_points.extend(page.get("dataPoints", []))

        next_page_token = page.get("nextPageToken", "")
        if not next_page_token:
            return {"dataPoints": all_points, "nextPageToken": ""}
        if next_page_token in seen_page_tokens:
            raise RuntimeError(
                "Google Health repeated a nextPageToken; pagination stopped."
            )
        seen_page_tokens.add(next_page_token)
        page_token = next_page_token


def _step_points(response):
    '''Return a validated raw step DataPoint list; [] means no observations.'''
    if not isinstance(response, dict):
        raise TypeError("A step response must be a dictionary wrapper.")
    points = response.get("dataPoints", [])
    if not isinstance(points, list):
        raise TypeError("A step response dataPoints field must be a list.")
    return points


def step_application_package(point):
    '''Return application.packageName, or None when attribution is absent.'''
    data_source = point.get("dataSource")
    if not isinstance(data_source, dict):
        return None
    application = data_source.get("application")
    if not isinstance(application, dict):
        return None
    package_name = application.get("packageName")
    if not isinstance(package_name, str) or not package_name:
        return None
    return package_name


def raw_step_total(response):
    '''Return one already-filtered raw sum, or None for no observations.'''
    points = _step_points(response)
    if not points:
        return None
    # Google may omit the default-valued count for an on-wrist true-zero point.
    return sum(int(point["steps"].get("count", "0")) for point in points)


def step_source_report(response):
    '''Return separate package totals and individual unattributed points.'''
    attributed = {}
    unattributed = []
    for index, point in enumerate(_step_points(response)):
        count = int(point["steps"].get("count", "0"))
        package_name = step_application_package(point)
        if package_name is None:
            unattributed.append({"point_index": index, "count": count})
            continue
        facts = attributed.setdefault(
            package_name,
            {"point_count": 0, "raw_step_sum": 0},
        )
        facts["point_count"] += 1
        facts["raw_step_sum"] += count
    return dict(sorted(attributed.items())), unattributed


def print_step_source_report(response):
    '''Print source-separated raw results without an all-source total.'''
    attributed, unattributed = step_source_report(response)
    if not _step_points(response):
        print("No step observations are present.")
        return
    print("Package-attributed step sources:")
    if not attributed:
        print("- none")
    for package_name, facts in attributed.items():
        print(
            f"- {package_name}: {facts['point_count']} point(s), "
            f"raw per-source sum {facts['raw_step_sum']}"
        )
    for facts in unattributed:
        print(
            f"- point index {facts['point_index']}: count {facts['count']} "
            "is unattributed and excluded because application.packageName "
            "is absent"
        )


def choose_step_application_source(
    response,
    requested_source="",
    control_name="SELECTED_STEP_APPLICATION_SOURCE",
):
    '''Validate one exact package source; auto-select only a sole source.'''
    points = _step_points(response)
    attributed, _ = step_source_report(response)
    valid_sources = list(attributed)
    if not points:
        return None
    if not valid_sources:
        raise ValueError(
            "Step DataPoints are present, but none has a nonempty "
            "dataSource.application.packageName. No source-specific total "
            "can be computed. Choose another date/path or inspect the raw "
            "points without aggregating them."
        )
    if not isinstance(requested_source, str):
        raise ValueError(f"{control_name} must be a string.")
    if requested_source:
        if requested_source not in valid_sources:
            choices = ", ".join(repr(source) for source in valid_sources)
            raise ValueError(
                f"{control_name} must exactly match one of: {choices}"
            )
        return requested_source
    if len(valid_sources) == 1:
        return valid_sources[0]
    choices = ", ".join(repr(source) for source in valid_sources)
    raise ValueError(
        f"Multiple step application sources are present. Set {control_name} "
        f"to one exact value from the source report: {choices}"
    )


def filter_steps_by_application_source(response, selected_source):
    '''Return a wrapper containing only one named application source.'''
    points = _step_points(response)
    filtered = copy.deepcopy(response)
    if not points:
        filtered["dataPoints"] = []
        return filtered
    valid_sources = {
        source
        for point in points
        if (source := step_application_package(point)) is not None
    }
    if selected_source not in valid_sources:
        choices = ", ".join(repr(source) for source in sorted(valid_sources))
        raise ValueError(
            "selected_source must exactly match a package-attributed step "
            f"source. Valid sources: {choices or 'none'}"
        )
    filtered["dataPoints"] = [
        copy.deepcopy(point)
        for point in points
        if step_application_package(point) == selected_source
    ]
    return filtered


print(
    "Google Health client ready. Use force_sample=True for the guaranteed "
    "synthetic path or force_sample=False for your authorized private data."
)

---

## Part 1 — Get data from the Fitbit API

Start with `USE_FITBIT_DATA=False`: this loads the synthetic **2026-09-24**
records. Orson will switch it to `True` to demonstrate his **2026-09-10**
Fitbit data. The live-date control applies only when that switch is on.
When your own device has history, choose a date you wore and synced it, then
select `True` to request your private records.

One date gives us three responses: step intervals, heart-rate samples, and
sleep sessions ending on that local date. The cell prints their sizes. An
empty response means **no observations**, not a measured zero. The API route
is `GET /v4/users/me/dataTypes/{type}/dataPoints`; the client returns all pages.

In [ ]:
#@title Provided - Part 1.1 - Choose the class path and fetch data
# Choose the data path for Parts 1–2. The live date is for Orson's demo;
# students using their own Fitbit history should select their own synced date.
USE_FITBIT_DATA = False  # @param {type:"boolean"}
FITBIT_DATA_DATE = "2026-09-10"  # @param {type:"date"}
SELECTED_DATA_DATE = FITBIT_DATA_DATE if USE_FITBIT_DATA else "2026-09-24"

# Invalidate old results before fetching, including partially completed runs.
steps_data = heart_rate_data = sleep_data = None
heart_rate_points = sleep_points = None
selected_steps_data = selected_source_step_total = None
ACTIVE_USE_FITBIT_DATA = ACTIVE_DATA_DATE = None
ACTIVE_STEP_APPLICATION_SOURCE = None
PART1_FETCH_READY = PART1_SOURCE_READY = False
step_figure = heart_rate_figure = sleep_figure = None
_class_candidate = None

_class_candidate = {
    data_type: list_health_data(
        data_type, SELECTED_DATA_DATE, force_sample=not USE_FITBIT_DATA
    )
    for data_type in ("steps", "heart-rate", "sleep")
}
# Publish all three results only after all three requests succeeded.
steps_data = _class_candidate["steps"]
heart_rate_data = _class_candidate["heart-rate"]
sleep_data = _class_candidate["sleep"]
heart_rate_points = heart_rate_data.get("dataPoints", [])
sleep_points = sleep_data.get("dataPoints", [])
ACTIVE_USE_FITBIT_DATA = bool(USE_FITBIT_DATA)
ACTIVE_DATA_DATE = SELECTED_DATA_DATE
PART1_FETCH_READY = True

print("Data path:", "Fitbit" if ACTIVE_USE_FITBIT_DATA else "synthetic")
print("Local date:", ACTIVE_DATA_DATE)
for data_type, response in _class_candidate.items():
    count = len(response.get("dataPoints", []))
    print(f"{data_type}: {count} observations")
    if count == 0:
        print("  No observations; this does not establish a measured zero.")

### Provided - Part 1.2 — Choose one step source

Two applications can write overlapping step records. Run the source report,
then choose `com.google.fitbit` in the next cell. The sample also contains a
fictional imported stream: adding both sources would double-count activity.

The code keeps only the selected source before adding counts. Unattributed
points are excluded. This produces a **raw per-source sum**, not an official
reconciled daily total. Leave the source blank only when the report lists
exactly one package. Live data may use different package names.

In [ ]:
#@title Provided - Part 1.2 - Inspect step sources
def require_current_class_data():
    'Stop if the selected path or live date changed after the Part 1 fetch.'
    current_day = FITBIT_DATA_DATE if USE_FITBIT_DATA else "2026-09-24"
    if (
        not PART1_FETCH_READY
        or ACTIVE_USE_FITBIT_DATA != bool(USE_FITBIT_DATA)
        or ACTIVE_DATA_DATE != current_day
    ):
        raise RuntimeError("Rerun the Part 1 fetch for the current path and date.")


require_current_class_data()
print_step_source_report(steps_data)

In [ ]:
#@title Provided - Part 1.2 - Select one step source
SELECTED_STEP_APPLICATION_SOURCE = "com.google.fitbit"  # @param {type:"string"}

selected_steps_data = selected_source_step_total = None
ACTIVE_STEP_APPLICATION_SOURCE = None
PART1_SOURCE_READY = False
require_current_class_data()

selected_step_application_source = choose_step_application_source(
    steps_data, SELECTED_STEP_APPLICATION_SOURCE,
    "SELECTED_STEP_APPLICATION_SOURCE",
)
SELECTED_STEP_APPLICATION_SOURCE = selected_step_application_source or ""
selected_steps_data = filter_steps_by_application_source(
    steps_data, selected_step_application_source
)
selected_source_step_total = raw_step_total(selected_steps_data)
ACTIVE_STEP_APPLICATION_SOURCE = selected_step_application_source
PART1_SOURCE_READY = True

print("Selected source:", selected_step_application_source or "no observations")
print("Selected step intervals:", len(selected_steps_data["dataPoints"]))
print("Raw per-source sum:", selected_source_step_total)
# Default sample: 144 selected ten-minute intervals and 6,815 steps.
assert all(
    step_application_package(point) == ACTIVE_STEP_APPLICATION_SOURCE
    for point in selected_steps_data["dataPoints"]
)

---

## Part 2 — Simple visualizations

Start with the worked step chart. Then complete **one plotting line** for
heart rate and one for sleep. Run the provided preparation cell first: its helpers
convert API values, sort timestamps, and handle missing observations.

All three plots use the path and local date you loaded in Part 1. The
heart-rate line connects the available samples; it does not establish what
happened between them. The sleep bars describe one reported session, not a
clinical assessment of sleep quality.

In [ ]:
#@title Provided - Part 2 - Prepare plotting tables
def add_google_local_timestamp(
    df, physical_column, offset_column, local_column
):
    '''Add local timestamps derived from Google physical times and offsets.'''
    df[physical_column] = pd.to_datetime(df[physical_column], format="mixed", utc=True)
    df[local_column] = (
        df[physical_column] + pd.to_timedelta(df[offset_column])
    ).dt.tz_localize(None)


def prepare_heart_rate_df(response):
    '''Return a sorted heart-rate DataFrame, or None when it is empty.'''
    points = response.get("dataPoints", [])
    if not points:
        return None

    df = pd.DataFrame.from_records([
        {
            "physical_time": point["heartRate"]["sampleTime"][
                "physicalTime"
            ],
            "utc_offset": point["heartRate"]["sampleTime"]["utcOffset"],
            "beats_per_minute": int(point["heartRate"]["beatsPerMinute"]),
        }
        for point in points
    ])
    add_google_local_timestamp(
        df, "physical_time", "utc_offset", "local_time"
    )
    return df.sort_values("physical_time", kind="stable").reset_index(
        drop=True
    )


def summarize_sleep(response):
    '''Return a compact sleep dictionary, or None for no session.'''
    for point in response.get("dataPoints", []):
        sleep = point["sleep"]
        summary = sleep.get("summary") or {}
        if "minutesAsleep" in summary and "minutesAwake" in summary:
            return {
                "minutes_asleep": int(summary["minutesAsleep"]),
                "minutes_awake": int(summary["minutesAwake"]),
                "processed": sleep.get("metadata", {}).get("processed", False),
            }
    return None


SLEEP_STAGE_LABELS = {
    "AWAKE": "Awake", "LIGHT": "Light", "DEEP": "Deep", "REM": "REM",
    "RESTLESS": "Restless", "ASLEEP": "Asleep (classic)",
}
SLEEP_STAGE_COLORS = {
    "AWAKE": "#E67E22", "LIGHT": "#71A6D2", "DEEP": "#1F4E79",
    "REM": "#8C6BB1", "RESTLESS": "#D9A441", "ASLEEP": "#3B7D82",
}


def sleep_stage_label(stage):
    return SLEEP_STAGE_LABELS.get(stage, f"Unrecognized stage: {stage}")


def reported_sleep_stage_minutes(response):
    '''Return only reported stage durations from the first available stage summary.'''
    for point in response.get("dataPoints", []):
        entries = (point["sleep"].get("summary") or {}).get("stagesSummary", [])
        if entries:
            totals = {}
            for entry in entries:
                stage = str(entry.get("type") or "UNKNOWN")
                minutes = int(entry.get("minutes", "0"))
                if minutes < 0:
                    raise ValueError("Reported stage durations must be nonnegative.")
                totals[stage] = totals.get(stage, 0) + minutes
            return totals
    return None

### Provided - Part 2.1 — Worked example: step intervals

Run the next cell. Each bar shows the raw count for one interval from the
selected application source. The horizontal axis uses local clock time. In
the default sample, 144 bars cover the day in ten-minute intervals. The largest
bar begins at 18:50 and represents 668 steps. Bar widths follow the actual
interval durations, including when you use live data.
Use this chart as a model for the next two plotting lines.

In [ ]:
#@title Provided - Part 2.1 - Worked step-interval chart
def plot_step_counts(response, day, application_source):
    '''Plot raw step counts by interval for one filtered application source.'''
    points = response.get("dataPoints", [])
    if not points:
        print(f"No step observations are available to plot for {day}.")
        return None

    df = pd.DataFrame.from_records([
        {
            "physical_start": point["steps"]["interval"]["startTime"],
            "physical_end": point["steps"]["interval"]["endTime"],
            "start_utc_offset": point["steps"]["interval"][
                "startUtcOffset"
            ],
            "step_count": int(point["steps"].get("count", "0")),
        }
        for point in points
    ])
    add_google_local_timestamp(
        df, "physical_start", "start_utc_offset", "local_start"
    )
    df["physical_end"] = pd.to_datetime(df["physical_end"], format="mixed", utc=True)
    # Matplotlib date widths are in days; retain each interval's duration.
    df["interval_width_days"] = (
        df["physical_end"] - df["physical_start"]
    ).dt.total_seconds() / 86400
    df = df.sort_values("physical_start", kind="stable").reset_index(
        drop=True
    )

    figure, axis = plt.subplots(figsize=(9, 4))
    axis.bar(
        df["local_start"],
        df["step_count"],
        width=df["interval_width_days"],
        align="edge",
        color="#2E75B6",
    )
    axis.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    axis.set_xlabel("Interval time (local clock)")
    axis.set_ylabel("Raw step count")
    axis.set_title(f"{day}: raw steps from {application_source}")
    axis.grid(axis="y", alpha=0.25)
    axis.set_ylim(bottom=0)
    figure.autofmt_xdate(rotation=30, ha="right")
    figure.tight_layout()
    plt.show()
    return figure


require_current_class_data()
if (
    not PART1_SOURCE_READY
    or SELECTED_STEP_APPLICATION_SOURCE != (ACTIVE_STEP_APPLICATION_SOURCE or "")
):
    raise RuntimeError("Complete Part 1 before plotting.")
part2_day = ACTIVE_DATA_DATE
step_figure = plot_step_counts(
    selected_steps_data,
    ACTIVE_DATA_DATE,
    ACTIVE_STEP_APPLICATION_SOURCE,
)

### TODO - Part 2.2 — heart rate

The supplied `prepare_heart_rate_df` returns a table with `local_time` and
`beats_per_minute`. In `plot_heart_rate`, replace the marked `raise` line with
one `axis.plot(...)` call. Use those two columns, `marker="o"`, and
`markersize=3` so the points remain readable. The axis
labels and layout are already provided. Run the next check cell.

**Expected:** 144 point samples, ten minutes apart, ranging from 57 to 118 bpm
for the default sample. Each value is a point measurement, not a ten-minute mean.
An empty successful response prints an unavailable message and draws no plot.

In [ ]:
#@title TODO - Part 2.2 - Plot heart-rate points
def plot_heart_rate(response, day):
    '''Plot beats per minute over the selected local day.'''
    df = prepare_heart_rate_df(response)
    if df is None:
        print(f"No heart-rate observations are available to plot for {day}.")
        return None

    figure, axis = plt.subplots(figsize=(8, 4))

    # TODO: Plot beats_per_minute against local_time with small circle markers.
    # Hint: Think about which column belongs on each axis, then choose a
    # plotting method that shows how a measurement changes over time.
    raise NotImplementedError("Implement plot_heart_rate: replace this line")
    axis.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    axis.set_xlabel("Sample time (local clock)")
    axis.set_ylabel("Beats per minute")
    axis.set_title(f"{day}: heart-rate samples")
    axis.grid(alpha=0.25)
    figure.autofmt_xdate(rotation=30, ha="right")
    figure.tight_layout()
    plt.show()
    return figure


require_current_class_data()
part2_day = ACTIVE_DATA_DATE
heart_rate_figure = plot_heart_rate(heart_rate_data, part2_day)

In [ ]:
#@title Provided - Part 2.2 - Check the heart-rate plot
# Verify the preparation helper, plot, and empty valid response.
heart_rate_df = prepare_heart_rate_df(heart_rate_data)
if heart_rate_points:
    assert list(heart_rate_df.columns) == [
        "physical_time", "utc_offset", "beats_per_minute", "local_time"
    ]
    assert heart_rate_df["physical_time"].is_monotonic_increasing
    assert heart_rate_figure is not None
    assert len(heart_rate_figure.axes[0].lines[0].get_ydata()) == len(
        heart_rate_points
    )
    print(f"Plotted {len(heart_rate_points)} heart-rate samples.")
else:
    assert heart_rate_df is None
    assert heart_rate_figure is None
    print("Heart-rate values are unavailable for the selected path and date.")
assert prepare_heart_rate_df({"dataPoints": []}) is None
assert plot_heart_rate({"dataPoints": []}, part2_day) is None
print("Part 2.2 checks passed.")

### TODO - Part 2.3 — sleep stages

The supplied `reported_sleep_stage_minutes` reads the first available
`summary.stagesSummary`: each entry reports a stage `type` and its `minutes`.
Replace the marked `raise` line with a bar chart using the prepared labels
and values. Compare the reported categories; do not add the aggregate asleep
or awake totals again. Classic sleep records may report different categories.
Unknown stage labels stay visible, and a missing stage summary is unavailable.

**Expected:** Awake 40, Light 228, Deep 83, and REM 103 minutes for the authored
class sample. Light + Deep + REM equals its aggregate 414 minutes asleep.
These are designed teaching durations, not measurements from a real night.

Reference: [Google Health sleep fields](https://developers.google.com/health/reference/rest/v4/users.dataTypes.dataPoints#Sleep).

In [ ]:
#@title TODO - Part 2.3 - Plot reported sleep-stage durations
def plot_sleep_summary(response, day):
    '''Plot each reported stage duration for one selected sleep session.'''
    stages = reported_sleep_stage_minutes(response)
    if stages is None:
        print(f"No stage-duration summary is available to plot for {day}.")
        return None
    labels = [sleep_stage_label(stage) for stage in stages]
    values = list(stages.values())

    figure, axis = plt.subplots(figsize=(6, 4))

    # TODO: Draw one bar for every reported sleep-stage category.
    # Hint: Match each category label to its duration; choose a plotting
    # method that compares separate categories.
    raise NotImplementedError("Implement plot_sleep_summary: replace this line")
    axis.set_ylabel("Minutes")
    axis.set_title(f"{day}: reported sleep-stage durations")
    axis.grid(axis="y", alpha=0.25)
    figure.tight_layout()
    plt.show()
    return figure


require_current_class_data()
part2_day = ACTIVE_DATA_DATE
sleep_figure = plot_sleep_summary(sleep_data, part2_day)

In [ ]:
#@title Provided - Part 2.3 - Check the sleep-stage plot
# Verify the reported categories and missing-stage-summary branch.
sleep_stage_minutes = reported_sleep_stage_minutes(sleep_data)
if sleep_stage_minutes is not None:
    assert sleep_figure is not None
    assert all(value >= 0 for value in sleep_stage_minutes.values())
    assert [bar.get_height() for bar in sleep_figure.axes[0].patches] == list(sleep_stage_minutes.values())
    print("Plotted every reported stage-duration category.")
else:
    assert sleep_figure is None
    print("Stage durations are unavailable; aggregate totals cannot supply them.")
assert reported_sleep_stage_minutes({"dataPoints": []}) is None
assert plot_sleep_summary({"dataPoints": []}, part2_day) is None
assert reported_sleep_stage_minutes({"dataPoints": [{"sleep": {"summary": {
    "minutesAsleep": "414", "minutesAwake": "40"
}}}]}) is None
assert reported_sleep_stage_minutes({"dataPoints": [{"sleep": {"summary": {
    "stagesSummary": [{"type": "AWAKE"}, {"type": "FUTURE_STAGE", "minutes": "2"}]
}}}]}) == {"AWAKE": 0, "FUTURE_STAGE": 2}
print("Part 2.3 checks passed.")

---

## Part 3 — Create and check an AI summary

Choose a focus for a short summary, inspect the evidence, and check one claim.
Use your issued OpenAI API key to generate
a summary of the selected data, then check it against the evidence.

### Provided - Part 3.1 — Choose the data and inspect the evidence

Keep the data defaults: **synthetic, 2026-09-24,
`com.google.fitbit`**. This is the same small sample used in class, not the
detailed homework fixture. Part 3 has a separate live-data choice so a live
demonstration in Part 1 does not also authorize an OpenAI transfer.
Its live-date control also starts at **2026-09-10** for Orson's demonstration;
choose your own synced date if you select your private Fitbit records.

The next cell prepares the selected responses and prints a compact evidence
table. When you run the request cell,
the **full source-filtered responses** go to OpenAI. These can contain exact
timestamps, DataPoint names, and device/source metadata beyond the table.
Google credentials are never included. OAuth and the course helper send
nothing to OpenAI. The request uses `store=False`; that is not a guarantee of
zero retention. A failed call stops without substituting another response.

Reference: [OpenAI Python quickstart](https://developers.openai.com/api/docs/quickstart).

In [ ]:
#@title Provided - Part 3.1 - Choose model data and inspect evidence
# Part 3 has its own explicit data choice and date; sample is the default.
USE_FITBIT_FOR_OPENAI = False  # @param {type:"boolean"}
OPENAI_FITBIT_DATE = "2026-09-10"  # @param {type:"date"}
OPENAI_DATA_DATE = OPENAI_FITBIT_DATE if USE_FITBIT_FOR_OPENAI else "2026-09-24"
# Leave blank to auto-select only when steps have exactly one package source.
OPENAI_STEP_APPLICATION_SOURCE = "com.google.fitbit"  # @param {type:"string"}

# Invalidate any earlier displayed payload before a new fetch begins.
openai_selected_data = None
openai_displayed_json = None
OPENAI_DISPLAY_SELECTION = None
OPENAI_DISPLAY_READY = False
OPENAI_SELECTED_STEP_SOURCE = None
summary_response = None
generated_summary = None
summary_for_review = None
summary_origin = None
summary_evidence = None
_openai_raw_candidate_data = None
_openai_candidate_data = None
_openai_candidate_json = None

_openai_raw_candidate_data = {
    data_type: list_health_data(
        data_type,
        OPENAI_DATA_DATE,
        force_sample=not USE_FITBIT_FOR_OPENAI,
    )
    for data_type in ("steps", "heart-rate", "sleep")
}
print_step_source_report(_openai_raw_candidate_data["steps"])
OPENAI_SELECTED_STEP_SOURCE = choose_step_application_source(
    _openai_raw_candidate_data["steps"],
    OPENAI_STEP_APPLICATION_SOURCE,
    "OPENAI_STEP_APPLICATION_SOURCE",
)
# Store the resolved value so changing or rerunning the source control cannot
# reuse a payload displayed for a different package.
OPENAI_STEP_APPLICATION_SOURCE = OPENAI_SELECTED_STEP_SOURCE or ""
_openai_candidate_data = dict(_openai_raw_candidate_data)
_openai_candidate_data["steps"] = filter_steps_by_application_source(
    _openai_raw_candidate_data["steps"],
    OPENAI_SELECTED_STEP_SOURCE,
)
_openai_candidate_json = json.dumps(_openai_candidate_data, indent=2)

print(
    "Part 3 data path:",
    "Fitbit" if USE_FITBIT_FOR_OPENAI else "synthetic",
)
print("Part 3 date:", OPENAI_DATA_DATE)
print(
    "Selected step application source:",
    OPENAI_SELECTED_STEP_SOURCE or "no step observations",
)
_summary_hr = prepare_heart_rate_df(_openai_candidate_data["heart-rate"])
summary_evidence = {
    "raw_selected_source_steps": raw_step_total(_openai_candidate_data["steps"]),
    "heart_rate_samples": 0 if _summary_hr is None else len(_summary_hr),
    "heart_rate_min_bpm": None if _summary_hr is None else int(_summary_hr["beats_per_minute"].min()),
    "heart_rate_max_bpm": None if _summary_hr is None else int(_summary_hr["beats_per_minute"].max()),
    "sleep_session": summarize_sleep(_openai_candidate_data["sleep"]),
}
print("Evidence for your summary:")
print(json.dumps(summary_evidence, indent=2))

openai_selected_data = _openai_candidate_data
openai_displayed_json = _openai_candidate_json
OPENAI_DISPLAY_SELECTION = (
    bool(USE_FITBIT_FOR_OPENAI),
    OPENAI_DATA_DATE,
    OPENAI_SELECTED_STEP_SOURCE,
)
OPENAI_DISPLAY_READY = True

### TODO - Part 3.2 — Choose a focus and request a summary

Write one sentence in `summary_focus` about a feature in the Part 3 evidence.
You may use your Part 2 plots if their path, date, and step source match the
independent Part 3 selection. Otherwise, choose your focus from the Part 3
evidence table. The request code supplies the date, source rule, and uncertainty
boundaries.

The **system prompt** (`openai_system_prompt`) defines the task and rules.
The **user prompt** (`openai_user_prompt`) supplies your focus and the selected
data. The system prompt and the first 1,000 characters of the user prompt are displayed
before the request; the complete user prompt is still sent. In the Responses API,
we send them as `instructions` and `input`, respectively.
See the [message-to-Responses mapping](https://developers.openai.com/api/docs/guides/migrate-to-responses#2-map-messages-to-items).

Every student has already received a key: open Colab **Secrets** (the key
icon), save your issued key under the exact name `OPENAI_API_KEY`, and enable
notebook access. Never paste it into a cell or share it. Run the setup below;
it checks availability without printing the key. Run the prompt/cost preview cell,
review its output, then run the separate request cell. A missing key or provider
error stops the request; fix the access problem and rerun it.

The preview counts both complete prompt strings locally. Its approximate input
cost uses the Standard uncached rate of **$0.20 per million input tokens**, doubled
above 272,000 estimated input tokens. It excludes API framing, generated output,
and cache adjustments. See [OpenAI pricing](https://developers.openai.com/api/docs/pricing)
(verified September 23, 2026). Previewing makes no model request.

In [ ]:
#@title TODO - Part 3.2 - Choose your summary focus
# TODO: Choose a focus from Part 3 evidence, or plots with matching path/date/source.
summary_focus = "TODO: name a pattern and what the summary should explain."

In [ ]:
#@title Provided - Part 3.2 - Read your issued OpenAI key
def read_private_secret(name):
    'Read a Colab Secret or local environment variable without printing it.'
    value = os.environ.get(name)
    if value:
        return value
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None


openai_client = None
summary_response = generated_summary = summary_for_review = None
summary_origin = None
openai_api_key = read_private_secret("OPENAI_API_KEY")
print("OPENAI_API_KEY:", "available" if openai_api_key else "not available")
if openai_api_key:
    openai_client = OpenAI(api_key=openai_api_key, timeout=30.0, max_retries=0)
print("No model request was made by this setup cell.")

In [ ]:
#@title Provided - Part 3.2 - Prompt and cost helpers
SUMMARY_MAX_OUTPUT_TOKENS = 500


def display_openai_prompts(system_prompt, user_prompt):
    '''Show the full system prompt and a labeled 1,000-character user preview.'''
    for label, prompt in (
        ("System prompt (`instructions`)", system_prompt),
        (f"User prompt preview (`input`; {min(1000, len(user_prompt)):,} of "
         f"{len(user_prompt):,} characters shown; full text is sent)", user_prompt[:1000]),
    ):
        fence = "```"
        while fence in prompt:
            fence += "`"
        display(Markdown(f"### {label}\n\n{fence}text\n{prompt}\n{fence}"))


def build_summary_instructions(day, step_application_source, task="Summarize the wearable data"):
    if step_application_source is None:
        step_source_rule = (
            "No step observations are available. Report that absence; do not "
            "turn missing observations into a measured zero."
        )
    else:
        step_source_rule = (
            "The step records were filtered to one application source "
            "before display. Use only those records, keep the result labeled "
            "as a raw per-source sum, and do not combine application sources."
        )
    return f'''Task:
{task} for {day} in clear language.

Requirements:
- Report what the data shows for steps, heart rate, and sleep.
- {step_source_rule}
- Say when a data type is empty or missing.
- Describe observed variation without inferring its cause, an activity, or health status.
- Do not repeat DataPoint names, exact timestamps, or device/data-source metadata.
- Do not diagnose a condition or give medical advice.
'''.strip()


def estimate_openai_input(system_prompt, user_prompt):
    'Count both complete strings locally; this does not send either to OpenAI.'
    try:
        encoding = tiktoken.encoding_for_model("gpt-5.6-luna")
        tokenizer_note = f"tiktoken {encoding.name}"
    except KeyError:
        encoding = tiktoken.get_encoding("o200k_base")
        tokenizer_note = "approximate o200k_base fallback (model mapping unavailable)"
    system_tokens = len(encoding.encode(system_prompt, disallowed_special=()))
    user_tokens = len(encoding.encode(user_prompt, disallowed_special=()))
    input_tokens = system_tokens + user_tokens
    multiplier = 2 if input_tokens > 272_000 else 1
    rate = 0.20 * multiplier  # Standard uncached input USD per million tokens.
    return {
        "system_tokens": system_tokens, "user_tokens": user_tokens,
        "estimated_input_tokens": input_tokens,
        "uncached_input_usd_per_million": rate,
        "long_context_multiplier": multiplier,
        "estimated_input_cost_usd": input_tokens * rate / 1_000_000,
        "tokenizer": tokenizer_note,
    }


def preview_openai_request(system_prompt, user_prompt):
    display_openai_prompts(system_prompt, user_prompt)
    estimate = estimate_openai_input(system_prompt, user_prompt)
    display(Markdown(
        f"**Estimated prompt text tokens:** {estimate['estimated_input_tokens']:,}; "
        f"**Estimated input cost:** \${estimate['estimated_input_cost_usd']:.6f} USD at "
        f"\${estimate['uncached_input_usd_per_million']:.2f}/million uncached input tokens. "
        f"Counted both full prompts locally with {estimate['tokenizer']}. "
        "Approximation excludes API framing, output tokens, and cache adjustments. "
    ))
    return estimate


def require_reviewed_prompts(snapshot, context, system_prompt, user_prompt):
    if (
        not isinstance(system_prompt, str) or not isinstance(user_prompt, str)
        or snapshot != (context, system_prompt, user_prompt)
    ):
        raise RuntimeError("Prompts or inputs changed, or no preview ran. Rerun the preceding prompt/cost preview cell.")


def current_class_summary_context():
    current_day = OPENAI_FITBIT_DATE if USE_FITBIT_FOR_OPENAI else "2026-09-24"
    selected = (bool(USE_FITBIT_FOR_OPENAI), current_day, OPENAI_SELECTED_STEP_SOURCE)
    if (
        not OPENAI_DISPLAY_READY or not isinstance(openai_displayed_json, str)
        or not openai_displayed_json or OPENAI_DATA_DATE != current_day
        or OPENAI_DISPLAY_SELECTION != selected
        or OPENAI_STEP_APPLICATION_SOURCE != (OPENAI_SELECTED_STEP_SOURCE or "")
        or json.dumps(openai_selected_data, indent=2) != openai_displayed_json
    ):
        raise RuntimeError("Part 3.1 did not finish for the current Fitbit/date/source selection. Rerun Part 3.1 before previewing or sending.")
    if not summary_focus.strip() or summary_focus.startswith("TODO"):
        raise ValueError("Complete the summary_focus TODO before continuing.")
    return (selected, summary_focus, openai_displayed_json)

In [ ]:
#@title Provided - Part 3.2 - Preview prompts and estimated input cost
# Review this output before running the separate request cell.
OPENAI_PROMPT_PREVIEW = None
openai_input_estimate = None
summary_response = generated_summary = summary_for_review = None
summary_origin = None
openai_system_prompt = openai_user_prompt = None
openai_prompt_context = current_class_summary_context()
openai_system_prompt = build_summary_instructions(OPENAI_DATA_DATE, OPENAI_SELECTED_STEP_SOURCE)
openai_user_prompt = "Student focus:\n" + summary_focus + "\n\nSelected-day wearable data (JSON):\n" + openai_displayed_json
openai_input_estimate = preview_openai_request(openai_system_prompt, openai_user_prompt)
OPENAI_PROMPT_PREVIEW = (openai_prompt_context, openai_system_prompt, openai_user_prompt)

In [ ]:
#@title Provided - Part 3.2 - Send the reviewed summary request
# Run after reviewing the prompt preview and input-cost estimate above.
summary_response = generated_summary = summary_for_review = None
summary_origin = None
current_context = current_class_summary_context()
require_reviewed_prompts(
    globals().get("OPENAI_PROMPT_PREVIEW"), current_context,
    globals().get("openai_system_prompt"), globals().get("openai_user_prompt"),
)
if openai_client is None:
    raise RuntimeError(
        "OPENAI_API_KEY is unavailable. In Colab Secrets (the key icon), "
        "save your issued key as OPENAI_API_KEY, enable notebook access, "
        "and rerun the Part 3.2 setup cell. If access still fails, contact "
        "the course team privately."
    )
summary_response = openai_client.responses.create(
    model="gpt-5.6-luna", instructions=openai_system_prompt, input=openai_user_prompt,
    reasoning={"effort": "none"}, max_output_tokens=SUMMARY_MAX_OUTPUT_TOKENS, store=False,
)
generated_summary = summary_response.output_text
summary_for_review = generated_summary
summary_origin = "OpenAI response for the selected data"
print(summary_origin)
display(Markdown(summary_for_review))

### TODO - Part 3.3 — Check a claim against the evidence

Complete the three short responses below using your actual model output.
Quote one numerical claim and compare it with a named field/value in
`summary_evidence`. If the claim is wrong, state the mismatch and correct value.
If there is no numerical claim, say so and cite a relevant evidence field.
Explain one thing the records cannot establish and whether the response addressed
your chosen focus.

Before submission, remove any DataPoint name, exact timestamp, or device detail
that a model repeats. Course staff read your retained summary and written
responses. If you used Fitbit data, those materials remain derived from your
Google Health data even after raw outputs are cleared.

In [ ]:
#@title TODO - Part 3.3 - Check the summary against evidence
# TODO: Check one claim using the evidence table, then assess its limits.
checked_claim = "TODO: quote a numerical claim, or state that none was provided."
supporting_evidence = "TODO: name the evidence field/value; state a match, mismatch, or relevant omitted value."
summary_comparison = (
    "TODO: state one unsupported inference or missing detail, and whether "
    "the summary addresses your summary_focus."
)
display(Markdown(checked_claim))
display(Markdown(supporting_evidence))
display(Markdown(summary_comparison))

---

## Part 4 — Take-home: build a selected-day summary

**Homework.** If you received a Fitbit, complete this assignment using your
own authorized data. Choose any date you wore and synced the device that
provides steps, heart rate, and sleep for your analysis. Students without a
Fitbit use the provided sample path. Both paths have the same analytic requirements.

- **Received a Fitbit:** choose one day of your authorized data and retrieve its raw
  steps, heart-rate, and sleep list responses. The live date starts at
  2026-09-10 for Orson's demo; replace it with a date from your own device history.
- **No Fitbit:** download the detailed course sample below (currently 2026-09-10).
  It contains one step DataPoint per minute, one heart-rate DataPoint every five
  minutes, and one sleep session.

The fictional record represents one plausible day for a 24-year-old male
college student; that design context is not stored as extra JSON fields. It is
an example, not a normative baseline. Inspect how activity, heart rate, sleep,
and stationary minutes relate, and consider what one day cannot establish.

The provided loader downloads the three files from the
[public course data folder](https://github.com/personal-health-agent/binf4070-2026/tree/main/week-03/data)
when you choose the no-Fitbit path in Part 4.1. It reads and checks the actual
local date in those records, using the sleep session's end date. Fitbit users
fetch their own records without downloading the public files. Your analysis
will preserve actual step interval
lengths, compare raw heart-rate points with a trailing mean, and distinguish
reported sleep-stage intervals from aggregate summary durations. A supplied cell
collects the resulting tables into compact JSON without repeated device metadata.

Complete the numbered homework steps below: choose your day and source;
build three separate enhanced figures; prepare the model data; request an AI
day summary; and make a second request about a pattern from your plots.
Read both responses alongside your figures. Both OpenAI requests are required. Use the key
already issued to you. Your code may reuse class functions, but the analysis
and interpretation should reflect your chosen day.

Keep the three figures and both reviewed AI outputs in your submission,
with your final two reflections. Remove identifying
names, account details, and exact dates from figure titles and reviewed prose.
Clear raw records and prompt previews before submitting; never submit
credentials or another person's records.

In [ ]:
#@title Provided - Part 4.1 - Public homework data loader
HOMEWORK_PUBLIC_DATA_BASE = "https://raw.githubusercontent.com/personal-health-agent/binf4070-2026/main/week-03/data"
DETAILED_DAY_DATE = None  # Derived from the downloaded records, never relabeled.


def validate_public_homework_day(candidate):
    '''Require complete typed wrappers describing one consistent local day.'''
    dates = set()
    for kind, record_key in (("steps", "steps"), ("heart-rate", "heartRate"), ("sleep", "sleep")):
        wrapper = candidate[kind]
        if not isinstance(wrapper, dict) or not isinstance(wrapper.get("dataPoints"), list):
            raise ValueError(f"Public {kind} file must contain a dataPoints list.")
        if wrapper.get("nextPageToken") not in (None, ""):
            raise ValueError(f"Public {kind} file has an unconsumed page token.")
        for point in wrapper["dataPoints"]:
            try:
                if not isinstance(point, dict) or not isinstance(point.get(record_key), dict):
                    raise ValueError("Wrong record type")
                record = point[record_key]
                if kind == "heart-rate":
                    civil = record["sampleTime"]["civilTime"]
                    observed = pd.to_datetime(record["sampleTime"]["physicalTime"], format="mixed", utc=True, errors="raise")
                    if pd.isna(observed):
                        raise ValueError("Invalid sample time")
                    if int(record["beatsPerMinute"]) <= 0:
                        raise ValueError("Invalid heart rate")
                else:
                    interval = record["interval"]
                    civil = interval["civilStartTime" if kind == "steps" else "civilEndTime"]
                    start = pd.to_datetime(interval["startTime"], format="mixed", utc=True, errors="raise")
                    end = pd.to_datetime(interval["endTime"], format="mixed", utc=True, errors="raise")
                    if pd.isna(start) or pd.isna(end) or end <= start:
                        raise ValueError("Invalid interval")
                    if kind == "steps" and int(record.get("count", "0")) < 0:
                        raise ValueError("Invalid count")
                    if kind == "sleep":
                        for field in ("minutesAsleep", "minutesAwake"):
                            if field in (record.get("summary") or {}) and int(record["summary"][field]) < 0:
                                raise ValueError("Invalid sleep summary")
                date = civil["date"]
                dates.add(datetime(date["year"], date["month"], date["day"]).strftime("%Y-%m-%d"))
            except (KeyError, TypeError, ValueError, OverflowError) as error:
                raise ValueError(f"Public {kind} file has an invalid typed record or date.") from error
    if len(dates) != 1:
        raise ValueError("Public homework records must identify exactly one shared local day.")
    return dates.pop()


def load_public_homework_day():
    '''Download all three files; return only after every response validates.'''
    candidate = {}
    for kind in ("steps", "heart-rate", "sleep"):
        response = requests.get(f"{HOMEWORK_PUBLIC_DATA_BASE}/{kind}.json", timeout=30.0)
        response.raise_for_status()
        candidate[kind] = response.json()
    day = validate_public_homework_day(candidate)
    return day, candidate

In [ ]:
#@title Provided - Part 4.1 - Homework state checks
def current_takehome_selection():
    if not isinstance(takehome_use_fitbit, bool):
        raise ValueError("Choose takehome_use_fitbit=True or False in Part 4.1.")
    day = takehome_fitbit_date if takehome_use_fitbit else DETAILED_DAY_DATE
    if not day:
        raise RuntimeError("Run Part 4.1 to fetch the selected day first.")
    return (takehome_use_fitbit, day, takehome_step_application_source)


def invalidate_takehome_ai():
    for name in (
        "takehome_compact_data", "takehome_compact_json", "TAKEHOME_PAYLOAD_SELECTION",
        "takehome_response", "takehome_summary", "TAKEHOME_SUMMARY_INPUT",
        "takehome_pattern_response", "takehome_pattern_description",
        "TAKEHOME_SUMMARY_PREVIEW", "TAKEHOME_PATTERN_PREVIEW",
        "takehome_input_estimate", "takehome_pattern_input_estimate",
    ):
        globals()[name] = None


def require_current_takehome_data():
    if (
        not TAKEHOME_DATA_READY
        or TAKEHOME_DATA_SELECTION != current_takehome_selection()
    ):
        raise RuntimeError("Rerun Part 4.1 for the current path/date/source.")


def current_takehome_analysis():
    return (current_takehome_selection(), takehome_hr_window, takehome_hr_min_samples)


def require_current_takehome_analysis():
    require_current_takehome_data()
    selected = current_takehome_selection()
    if (
        TAKEHOME_STEPS_SELECTION != selected
        or TAKEHOME_SLEEP_SELECTION != selected
        or TAKEHOME_HR_SELECTION != current_takehome_analysis()
    ):
        raise RuntimeError("Complete Parts 4.2–4.4 for the current data and rolling window.")


def require_current_takehome_payload():
    require_current_takehome_analysis()
    if (
        TAKEHOME_PAYLOAD_SELECTION != current_takehome_analysis()
        or not takehome_compact_json
        or json.dumps(takehome_compact_data, indent=2) != takehome_compact_json
    ):
        raise RuntimeError("Rerun Part 4.5 to prepare the current model input.")


def current_takehome_summary_context():
    require_current_takehome_payload()
    if not takehome_summary_rules.strip() or takehome_summary_rules.startswith("TODO"):
        raise ValueError("Complete your Part 4.5 summary rules.")
    return (TAKEHOME_PAYLOAD_SELECTION, takehome_compact_json, takehome_summary_rules)


def current_takehome_pattern_context():
    summary_context = current_takehome_summary_context()
    if TAKEHOME_SUMMARY_INPUT != summary_context or takehome_summary is None:
        raise RuntimeError("Rerun Part 4.5 for the current data and summary rules.")
    if not takehome_pattern_focus.strip() or takehome_pattern_focus.startswith("TODO"):
        raise ValueError("Complete your Part 4.6 pattern focus.")
    return (summary_context, takehome_summary, takehome_pattern_focus)

def recorded_local_timestamp(physical_time, utc_offset):
    'Use the record offset; a missing offset is not a measured UTC offset of zero.'
    if utc_offset is None or pd.isna(utc_offset):
        raise ValueError("Recorded UTC offset is missing; local time is unavailable. Choose records with offsets.")
    offset = pd.to_timedelta(utc_offset)
    if pd.isna(offset):
        raise ValueError("Recorded UTC offset is missing; local time is unavailable.")
    zone = timezone(offset.to_pytimedelta())
    return pd.to_datetime(physical_time, format="mixed", utc=True).tz_convert(zone)


def recorded_local_iso(physical_time, utc_offset):
    'Return an offset-bearing local timestamp for a table or model payload.'
    return recorded_local_timestamp(physical_time, utc_offset).isoformat()


def format_recorded_local_axis(axis, physical_times, utc_offsets):
    'Keep elapsed-time geometry; label using only the offsets reported at observed anchors.'
    anchors = {}
    for physical, offset in zip(physical_times, utc_offsets):
        local = recorded_local_timestamp(physical, offset)
        seconds = local.utcoffset().total_seconds()
        instant = local.tz_convert("UTC")
        anchors[(instant.value, seconds)] = (instant, local, seconds)
    ordered = sorted(anchors.values(), key=lambda item: (item[0], item[2]))
    if not ordered:
        axis.set_xlabel("Local time unavailable: no observations")
        return
    offsets = {item[2] for item in ordered}
    if len(offsets) == 1:
        zone = ordered[0][1].tzinfo
        offset_text = ordered[0][1].strftime("%z")
        offset_text = offset_text[:3] + ":" + offset_text[3:]
        axis.xaxis.set_major_locator(mdates.AutoDateLocator(tz=zone, minticks=4, maxticks=7))
        axis.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M", tz=zone))
        axis.set_xlabel(f"Local clock time (recorded UTC{offset_text})")
        return
    # Keep both observed sides of each offset change; do not infer when it occurred.
    chosen = {round(i * (len(ordered) - 1) / 5) for i in range(6)}
    for index in range(1, len(ordered)):
        if ordered[index][2] != ordered[index - 1][2]:
            chosen.update((index - 1, index))
    tick_labels = {}
    for index in sorted(chosen):
        physical, local, _ = ordered[index]
        offset_text = local.strftime("%z")
        offset_text = offset_text[:3] + ":" + offset_text[3:]
        label = local.strftime("%H:%M") + f" UTC{offset_text}"
        tick_labels.setdefault(mdates.date2num(physical), []).append(label)
    axis.set_xticks(list(tick_labels), ["\n".join(labels) for labels in tick_labels.values()])
    axis.set_xlabel("Recorded local clock times and offsets; spacing shows elapsed time")

### TODO - Part 4.1 — Select your day and source

Set the path explicitly: `True` if you received a Fitbit, `False` if you did
not. For Fitbit, choose any worn and synced date that supports the analyses.
Explain why you chose that day. Inspect the source report and select one exact
step application package. The code fetches all three types before publishing
any result; errors stop the workflow. Changing a date, path, or source requires
rerunning this cell and the analyses below.

In [ ]:
#@title TODO - Part 4.1 - Choose your homework day and source
# TODO: Select your required path/date and explain your choice.
takehome_use_fitbit = None  # True: your Fitbit; False: no Fitbit, supplied sample.
takehome_fitbit_date = "2026-09-10"  # Replace with your own worn/synced date.
takehome_step_application_source = "com.google.fitbit"
takehome_date_reason = "TODO: why does this day support your analysis?"

In [ ]:
#@title Provided - Part 4.1 - Fetch and filter your homework day
TAKEHOME_DATA_READY = False
TAKEHOME_DATA_SELECTION = None
TAKEHOME_STEPS_SELECTION = TAKEHOME_HR_SELECTION = TAKEHOME_SLEEP_SELECTION = None
takehome_data = None
takehome_steps_df = takehome_hr_df = takehome_sleep_df = None
takehome_sleep_stages_df = takehome_sleep_timeline_df = takehome_sleep_timeline_source = None
takehome_sleep_timeline_sessions_df = None
takehome_steps_figure = takehome_hr_figure = takehome_sleep_figure = None
invalidate_takehome_ai()
_takehome_raw_candidate = None
_takehome_candidate = None
takehome_date = takehome_selected_step_source = None
DETAILED_DAY_DATE = None
if not isinstance(takehome_use_fitbit, bool):
    raise ValueError("Choose takehome_use_fitbit=True or False in Part 4.1.")
_candidate_day = takehome_fitbit_date if takehome_use_fitbit else None
if takehome_date_reason.startswith("TODO") or not takehome_date_reason.strip():
    raise ValueError("Explain your date choice in Part 4.1.")
if takehome_use_fitbit:
    _takehome_raw_candidate = {
        kind: list_health_data(kind, _candidate_day, force_sample=False)
        for kind in ("steps", "heart-rate", "sleep")
    }
else:
    _candidate_day, _takehome_raw_candidate = load_public_homework_day()
print_step_source_report(_takehome_raw_candidate["steps"])
takehome_selected_step_source = choose_step_application_source(
    _takehome_raw_candidate["steps"], takehome_step_application_source,
    "takehome_step_application_source",
)
takehome_step_application_source = takehome_selected_step_source or ""
_takehome_candidate = dict(_takehome_raw_candidate)
_takehome_candidate["steps"] = filter_steps_by_application_source(
    _takehome_raw_candidate["steps"], takehome_selected_step_source,
)
takehome_data = _takehome_candidate
takehome_date = _candidate_day
if not takehome_use_fitbit:
    DETAILED_DAY_DATE = _candidate_day
TAKEHOME_DATA_SELECTION = current_takehome_selection()
TAKEHOME_DATA_READY = True
print("Selected local date:", takehome_date)
print("Selected step source:", takehome_selected_step_source or "no observations")
for kind, wrapper in takehome_data.items():
    print(f"{kind}: {len(wrapper.get('dataPoints', [])):,} observations")
display(Markdown(takehome_date_reason))

### TODO - Part 4.2 — Interval steps and observed cumulative steps

Build `takehome_steps_df` from the selected source's records. Keep physical
`start`/`end` timestamps, `start_utc_offset`/`end_utc_offset`, and integer `count`.
Parse with `pd.to_datetime(..., format="mixed", utc=True)` and sort physically.
Add `local_start`/`local_end` with `recorded_local_iso`: these strings include
each endpoint's recorded offset. An absent offset cannot establish local time.

In `takehome_steps_figure`, make two panels: counts in their actual intervals,
and cumulative **observed** counts credited at interval ends. Widths and ordering
use physical time, even if local clocks change. The provided axis formatter
shows local time using recorded offsets, without assuming a geographic timezone.

Implement the record loop, table preparation, cumulative series, and plotting
calls below. The supplied coverage groups handle nested/overlapping intervals;
within each group, sort count contributions by end time. Start each curve with
the preceding observed total and leave gaps between groups. Missing intervals
are not zeros. Keep identifying dates out of titles; empty data gets an
unavailable annotation.

In [ ]:
#@title TODO - Part 4.2 - Compare interval and cumulative steps
require_current_takehome_data()
TAKEHOME_STEPS_SELECTION = None
invalidate_takehome_ai()


def prepare_takehome_steps(response):
    "Return physical/record-local endpoints, counts, coverage segments, and observed cumulative counts."
    columns = ["start", "end", "start_utc_offset", "end_utc_offset", "count"]
    rows = []
    for point in response.get("dataPoints", []):
        # TODO: 1. Read steps and its interval; append a dict matching columns.
        # Hint: counts are strings; an omitted count means zero. Keep both
        # endpoint offsets, and leave missing offsets as None rather than guessing.
        raise NotImplementedError("Extract one row per observed step interval")

    # TODO: 2. Build a table with columns even when rows is empty. Parse both
    # timestamps with format="mixed", utc=True; sort by start and reset the index.
    # Add local_start/local_end by pairing each endpoint with its own offset
    # through recorded_local_iso(time, offset). Keep physical columns unchanged.
    df = None
    raise NotImplementedError("Prepare the physical and recorded-local interval table")
    if not df.empty and (df["end"] <= df["start"]).any():
        raise ValueError("Step intervals must have a positive duration.")
    # Supplied: running coverage groups preserve gaps even with nested intervals.
    df["segment"] = (df["start"] > df["end"].cummax().shift()).cumsum()
    # TODO: 3. Sort by end, accumulate count, and align the result back to df.index.
    # Hint: cumsum() and reindex() preserve the link between counts and rows.
    raise NotImplementedError("Compute end-ordered cumulative observed counts")
    return df


takehome_steps_df = prepare_takehome_steps(takehome_data["steps"])
takehome_steps_figure, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
if takehome_steps_df.empty:
    for axis in axes:
        axis.text(0.5, 0.5, "No step observations", transform=axis.transAxes, ha="center")
else:
    df = takehome_steps_df
    # TODO: 4. Calculate physical widths in days and draw interval-count bars
    # on axes[0]. Hint: bar(..., align="edge") starts each bar at its timestamp.
    raise NotImplementedError("Plot actual step intervals")
    for _, segment in df.groupby("segment", sort=False):
        # TODO: 5. Sort by end, find the total before this group, and construct
        # time/total lists beginning at the group's earliest start. Draw a
        # separate axes[1].step(..., where="post") for this coverage group.
        raise NotImplementedError("Plot observed accumulation without bridging gaps")
axes[0].set_ylabel("Steps per observed interval")
axes[1].set_ylabel("Cumulative observed steps")
axes[0].set_title("Selected day: interval counts and observed accumulation")
for axis in axes:
    format_recorded_local_axis(
        axis, takehome_steps_df["start"].tolist() + takehome_steps_df["end"].tolist(),
        takehome_steps_df["start_utc_offset"].tolist() + takehome_steps_df["end_utc_offset"].tolist(),
    )
    axis.grid(alpha=0.2)
takehome_steps_figure.tight_layout()
plt.show()
TAKEHOME_STEPS_SELECTION = current_takehome_selection()

### TODO - Part 4.3 — Heart rate and a trailing mean

Choose `takehome_hr_window` (for example, `"30min"`) and
`takehome_hr_min_samples`, then explain the smoothing tradeoff in
`takehome_window_reason`. Prepare one row per recorded point with
`physical_time`, `utc_offset`, integer `beats_per_minute`, and offset-bearing
`local_time`. Use each point's own `sampleTime.utcOffset`; keep physical time
for sorting and rolling, and local time for interpretation.

Calculate `trailing_mean_bpm` and `window_sample_count` using a **time-based**
window. Include the current point and exclude a point exactly one window
before it. The mean requires your chosen minimum; the count reports all
observations in the window. Missing means should stay missing.

Implement the extraction/table preparation, rolling series, and both plotted
series in `takehome_hr_figure`. Show raw points and the trailing mean, labeled
with your window/minimum. Supplied gap groups prevent a line from connecting
samples farther apart than the window. The shared formatter labels recorded
local times while preserving elapsed spacing. Empty data produces an
unavailable figure, not fabricated points.

In [ ]:
#@title TODO - Part 4.3 - Compare raw and smoothed heart rate
require_current_takehome_data()
TAKEHOME_HR_SELECTION = None
invalidate_takehome_ai()
takehome_hr_window = "30min"  # TODO: choose a duration for your data.
takehome_hr_min_samples = 3  # TODO: choose the minimum observations.
takehome_window_reason = "TODO: explain your window/minimum and what smoothing may hide."

def prepare_takehome_heart_rate(response, window, minimum):
    "Return physical/record-local points, trailing means/counts, and gap segments."
    if pd.Timedelta(window) <= pd.Timedelta(0) or not isinstance(minimum, int) or minimum < 1:
        raise ValueError("Use a positive time window and integer minimum of at least one.")
    columns = ["physical_time", "utc_offset", "beats_per_minute"]
    rows = []
    for point in response.get("dataPoints", []):
        # TODO: 1. Read heartRate and its sampleTime; append the three named fields.
        # Hint: physicalTime and utcOffset belong to sampleTime; convert bpm to int.
        raise NotImplementedError("Extract one row per heart-rate point")
    if not rows:
        return pd.DataFrame(columns=columns + [
            "local_time", "trailing_mean_bpm", "window_sample_count", "gap_segment",
        ])
    # TODO: 2. Build the table, parse physical_time with format="mixed", utc=True,
    # sort it, and reset its index. Pair each point with its recorded offset
    # through recorded_local_iso to build the local_time strings.
    df = None
    raise NotImplementedError("Prepare the physical and recorded-local point table")

    # TODO: 3. Index bpm by physical_time. Use rolling(window, closed="right")
    # to calculate means and counts; choose min_periods separately for each.
    # Convert results to arrays when assigning to the row-indexed df columns.
    raise NotImplementedError("Calculate trailing_mean_bpm and window_sample_count")
    # Supplied: split the mean line across gaps longer than the chosen window.
    df["gap_segment"] = (df["physical_time"].diff() > pd.Timedelta(window)).cumsum()
    return df


takehome_hr_df = prepare_takehome_heart_rate(takehome_data["heart-rate"], takehome_hr_window, takehome_hr_min_samples)
takehome_hr_figure, axis = plt.subplots(figsize=(10, 4))
if takehome_hr_df.empty:
    axis.text(0.5, 0.5, "No heart-rate observations", transform=axis.transAxes, ha="center")
else:
    df = takehome_hr_df
    # TODO: 4. Scatter the raw physical-time/bpm pairs with a readable marker size.
    raise NotImplementedError("Draw and label the raw point series")
    for index, (_, segment) in enumerate(df.groupby("gap_segment", sort=False)):
        # TODO: 5. Draw this segment's trailing mean against physical time.
        # Label only the first segment, including your chosen window/minimum.
        raise NotImplementedError("Draw the trailing mean without bridging long gaps")
    axis.legend()
format_recorded_local_axis(axis, takehome_hr_df["physical_time"], takehome_hr_df["utc_offset"])
axis.set(ylabel="Beats per minute", title="Selected day: raw and smoothed heart rate")
axis.grid(alpha=0.2)
takehome_hr_figure.tight_layout()
plt.show()
display(Markdown(takehome_window_reason))
TAKEHOME_HR_SELECTION = current_takehome_analysis()

### TODO - Part 4.4 — Sleep-stage timeline

Keep selected-day session summaries (`takehome_sleep_df`) separate from their
reported stage intervals (`takehome_sleep_stages_df`). The supplied summary
function preserves session spans and aggregate minutes. Implement stage-table
preparation with these fields: `session_index`, physical `start`/`end`,
`start_utc_offset`/`end_utc_offset`, offset-bearing `local_start`/`local_end`,
`stage`, and physical `duration_minutes`.

Use each stage's own endpoint offsets, not an assumed timezone or the session's
offset. Parse with `pd.to_datetime(..., format="mixed", utc=True)`; sort physically
and keep unknown stage types visible. An overnight session can span two local
dates, and an offset change can repeat a clock hour. Durations use physical time.

Build the rows and sorted table, then draw the stage intervals in
`takehome_sleep_figure`. The supplied layout gives each session a panel and each
stage a lane. Implement the time-accurate plotting calls: position the bar at
its physical start and give it its elapsed width in days. The formatter labels
local times from session and stage endpoint offsets while preserving this
geometry. Leave unreported spans blank. Never reconstruct chronology from
summary totals.

**Missing stages:** Fitbit users get an unavailable message and should choose
another recorded night with stages. Live records are never supplemented. The
public no-Fitbit sample has only a summary, so the supplied routing uses a
separately labeled fictional **2026-09-24** session for timeline practice.
Its recorded offset is `0s`, so its local clock matches UTC; the actual public
September 10 records use `-14400s` (UTC−04:00). These are different sources.
The practice table stays separate from selected-day stages and model input;
choose your AI pattern from selected-day evidence.

In [ ]:
#@title TODO - Part 4.4 - Plot reported sleep-stage intervals
require_current_takehome_data()
TAKEHOME_SLEEP_SELECTION = None
invalidate_takehome_ai()

def prepare_takehome_sleep(response):
    '''Keep each actual session span and its available aggregate summary.'''
    rows = []
    for session_index, point in enumerate(response.get("dataPoints", []), start=1):
        session = point["sleep"]
        interval = session["interval"]
        start = pd.to_datetime(interval["startTime"], format="mixed", utc=True)
        end = pd.to_datetime(interval["endTime"], format="mixed", utc=True)
        start_offset = interval.get("startUtcOffset")
        end_offset = interval.get("endUtcOffset")
        summary = session.get("summary") or {}
        rows.append({
            "session_index": session_index,
            "start": start, "end": end,
            "start_utc_offset": start_offset, "end_utc_offset": end_offset,
            "local_start": recorded_local_iso(start, start_offset),
            "local_end": recorded_local_iso(end, end_offset),
            "duration_minutes": (end - start).total_seconds() / 60,
            "minutes_asleep": int(summary["minutesAsleep"]) if "minutesAsleep" in summary else None,
            "minutes_awake": int(summary["minutesAwake"]) if "minutesAwake" in summary else None,
        })
    return pd.DataFrame(rows, columns=[
        "session_index", "start", "end", "start_utc_offset", "end_utc_offset",
        "local_start", "local_end", "duration_minutes", "minutes_asleep", "minutes_awake",
    ]).sort_values("start", kind="stable").reset_index(drop=True)


def prepare_takehome_sleep_stages(response):
    "Flatten actual stage intervals with physical times and their own recorded-local endpoints."
    columns = [
        "session_index", "start", "end", "start_utc_offset", "end_utc_offset",
        "local_start", "local_end", "stage", "duration_minutes",
    ]
    rows = []
    for session_index, point in enumerate(response.get("dataPoints", []), start=1):
        for interval in point["sleep"].get("stages", []):
            # TODO: 1. Parse both physical timestamps with format="mixed", utc=True;
            # read each stage endpoint's own offset and preserve unknown types.
            # Hint: missing/empty type becomes "UNKNOWN"; an absent offset stays None.
            start = end = None
            raise NotImplementedError("Read the recorded stage interval and offsets")
            if pd.isna(start) or pd.isna(end) or end <= start:
                raise ValueError("Recorded sleep-stage intervals need valid increasing times.")
            # TODO: 2. Append a row matching columns. Calculate physical duration
            # in minutes and both local ISO strings with recorded_local_iso.
            raise NotImplementedError("Build one complete row per reported stage")

    # TODO: 3. Build a table with the declared schema even when rows is empty.
    # Sort by physical start/end, then reset the index; do not fill any gaps.
    raise NotImplementedError("Return the sorted stage-interval table")


takehome_sleep_df = prepare_takehome_sleep(takehome_data["sleep"])
takehome_sleep_stages_df = prepare_takehome_sleep_stages(takehome_data["sleep"])
takehome_sleep_timeline_df = takehome_sleep_stages_df.copy()
takehome_sleep_timeline_sessions_df = takehome_sleep_df.copy()
takehome_sleep_timeline_source = "Selected-day recorded sleep stages"
if takehome_sleep_stages_df.empty and not takehome_use_fitbit:
    practice_session = sample_list("sleep", "2026-09-24")
    takehome_sleep_timeline_df = prepare_takehome_sleep_stages(practice_session)
    takehome_sleep_timeline_sessions_df = prepare_takehome_sleep(practice_session)
    takehome_sleep_timeline_source = "Separate fictional practice session: 2026-09-24"
    display(Markdown(
        "The public selected-day record has no stage intervals. This timeline uses "
        "the separate authored **2026-09-24** class session for practice. Its "
        "chronology is excluded from the selected-day model input."
    ))

sessions = list(takehome_sleep_timeline_sessions_df["session_index"])
takehome_sleep_figure, axes_grid = plt.subplots(
    max(1, len(sessions)), 1, figsize=(11, max(3, 2.5 * len(sessions))),
    sharex=False, squeeze=False,
)
axes = axes_grid[:, 0]
if not sessions:
    axes[0].text(
        0.5, 0.5, "Stage intervals unavailable; choose another recorded night.",
        transform=axes[0].transAxes, ha="center",
    )
else:
    for axis, session_index in zip(axes, sessions):
        session_span = takehome_sleep_timeline_sessions_df[
            takehome_sleep_timeline_sessions_df["session_index"] == session_index
        ].iloc[0]
        intervals = takehome_sleep_timeline_df[
            takehome_sleep_timeline_df["session_index"] == session_index
        ]
        present = set(intervals["stage"])
        stages = [stage for stage in SLEEP_STAGE_LABELS if stage in present]
        stages += sorted(present - set(stages))
        lanes = {stage: index for index, stage in enumerate(stages)}
        if intervals.empty:
            axis.text(
                0.5, 0.5, "Stage intervals unavailable; choose another recorded night.",
                transform=axis.transAxes, ha="center",
            )
        for _, interval in intervals.iterrows():
            # TODO: 4. Compute the start coordinate, elapsed width in days,
            # and lane position; draw the interval with axis.broken_barh.
            # Hints: mdates.date2num converts the physical start; minutes/1440
            # gives width. Use the supplied lanes and colors (unknown = gray).
            # Leave unreported time blank; do not connect stages across a gap.
            raise NotImplementedError("Draw each time-accurate stage bar on its lane")
        axis.set_yticks(range(len(stages)), [sleep_stage_label(stage) for stage in stages])
        format_recorded_local_axis(
            axis,
            [session_span["start"], session_span["end"]] + intervals["start"].tolist() + intervals["end"].tolist(),
            [session_span["start_utc_offset"], session_span["end_utc_offset"]]
            + intervals["start_utc_offset"].tolist() + intervals["end_utc_offset"].tolist(),
        )
        axis.set_xlim(session_span["start"], session_span["end"])
        axis.set_title(f"Session {session_index}; may cross local midnight; blank spans have no reported stage")
if not sessions:
    axes[0].set_xlabel("Local time unavailable: no recorded sessions")
for axis in axes:
    axis.grid(axis="x", alpha=0.2)
takehome_sleep_figure.suptitle(takehome_sleep_timeline_source)
takehome_sleep_figure.tight_layout()
plt.show()
TAKEHOME_SLEEP_SELECTION = current_takehome_selection()

In [ ]:
#@title Provided - Part 4.5 - Collect selected-day model data
invalidate_takehome_ai()
require_current_takehome_analysis()

def compact_records(df, columns):
    rows = []
    for _, row in df.iterrows():
        compact = {}
        for key in columns:
            value = row[key]
            if pd.isna(value):
                compact[key] = None
            elif isinstance(value, pd.Timestamp):
                compact[key] = value.isoformat()
            elif hasattr(value, "item"):
                compact[key] = value.item()
            else:
                compact[key] = value
        rows.append(compact)
    return rows


takehome_compact_data = {
    "selected_local_date": takehome_date,
    "time_axis": "Plots show recorded-local clocks/offsets; local_* ISO values retain offsets. Physical UTC fields support elapsed durations, ordering, and rolling windows.",
    "steps": {
        "application_package_name": takehome_selected_step_source,
        "observed_count_sum": raw_step_total(takehome_data["steps"]),
        "interval_counts": compact_records(takehome_steps_df, [
            "start", "end", "start_utc_offset", "end_utc_offset", "local_start", "local_end", "count",
        ]),
    },
    "heart_rate": {
        "trailing_window": takehome_hr_window,
        "minimum_samples": takehome_hr_min_samples,
        "window_boundary": "exclude the left endpoint; include the current point",
        "samples": compact_records(takehome_hr_df, [
            "physical_time", "utc_offset", "local_time", "beats_per_minute", "trailing_mean_bpm", "window_sample_count",
        ]),
    },
    "sleep": {
        "sessions": compact_records(takehome_sleep_df, [
            "session_index", "start", "end", "start_utc_offset", "end_utc_offset", "local_start", "local_end",
            "duration_minutes", "minutes_asleep", "minutes_awake",
        ]),
        # Actual selected-day intervals only; separate practice chronology stays out.
        "stage_intervals": compact_records(takehome_sleep_stages_df, [
            "session_index", "start", "end", "start_utc_offset", "end_utc_offset", "local_start", "local_end",
            "stage", "duration_minutes",
        ]),
    },
}
takehome_compact_json = json.dumps(takehome_compact_data, indent=2)
TAKEHOME_PAYLOAD_SELECTION = current_takehome_analysis()
print("Model data prepared; both requests below preview their prompts; the complete input is sent.")

### TODO - Part 4.5 — Ask for a day summary

Write `takehome_summary_rules` to help the model distinguish observed counts
from missing intervals, raw heart-rate points from smoothed values, and a
sleep summary from actual stage intervals. Exclude any separate practice timeline
from your selected-day conclusions. The provided request adds your rules to
the class system prompt and supplies your complete compact data as the user
prompt. Review the separate prompt/cost preview, then run the request cell
using your issued key and read its rendered response.

In [ ]:
#@title TODO - Part 4.5 - Write day-summary rules
# TODO: Write evidence-specific rules for a useful, cautious day summary.
takehome_summary_rules = "TODO: what distinctions should the model preserve?"

In [ ]:
#@title Provided - Part 4.5 - Preview day-summary prompts and input cost
TAKEHOME_SUMMARY_PREVIEW = None
takehome_input_estimate = None
takehome_response = takehome_summary = TAKEHOME_SUMMARY_INPUT = None
takehome_pattern_response = takehome_pattern_description = None
TAKEHOME_PATTERN_PREVIEW = None
takehome_summary_context = current_takehome_summary_context()
takehome_system_prompt = build_summary_instructions(takehome_date, takehome_selected_step_source) + "\n" + takehome_summary_rules
takehome_user_prompt = "Summarize this selected day's observations.\n\n" + takehome_compact_json
takehome_input_estimate = preview_openai_request(takehome_system_prompt, takehome_user_prompt)
TAKEHOME_SUMMARY_PREVIEW = (takehome_summary_context, takehome_system_prompt, takehome_user_prompt)

In [ ]:
#@title Provided - Part 4.5 - Send the reviewed day-summary request
takehome_response = takehome_summary = TAKEHOME_SUMMARY_INPUT = None
takehome_pattern_response = takehome_pattern_description = None
current_context = current_takehome_summary_context()
require_reviewed_prompts(
    globals().get("TAKEHOME_SUMMARY_PREVIEW"), current_context,
    globals().get("takehome_system_prompt"), globals().get("takehome_user_prompt"),
)
if openai_client is None:
    raise RuntimeError("Add your issued OPENAI_API_KEY in Colab Secrets, enable notebook access, and rerun Part 3.2 setup.")
takehome_response = openai_client.responses.create(
    model="gpt-5.6-luna", instructions=takehome_system_prompt, input=takehome_user_prompt,
    reasoning={"effort": "none"}, max_output_tokens=SUMMARY_MAX_OUTPUT_TOKENS, store=False,
)
takehome_summary = takehome_response.output_text
TAKEHOME_SUMMARY_INPUT = current_context
display(Markdown(takehome_summary))

### TODO - Part 4.6 — Use AI to describe a pattern you found

Choose a feature supported by your selected-day records and figures, then state it in
`takehome_pattern_focus`. Ask the model to compare numerical evidence and
acknowledge a limitation, rather than invent an activity or cause. This is a
second request using the same selected data. A separately labeled practice sleep
timeline is from another day and must not supply this focus. Review the separate
prompt/cost preview before running the request cell; only the display is shortened. Compare its response with the plots and the first summary.

In [ ]:
#@title TODO - Part 4.6 - Choose an observed pattern
# TODO: Refer to a feature you actually observed; ask for a numerical comparison and a limit.
takehome_pattern_focus = "TODO: which pattern do you want the model to describe?"

In [ ]:
#@title Provided - Part 4.6 - Preview pattern prompts and input cost
TAKEHOME_PATTERN_PREVIEW = None
takehome_pattern_input_estimate = None
takehome_pattern_response = takehome_pattern_description = None
takehome_pattern_context = current_takehome_pattern_context()
takehome_pattern_system_prompt = build_summary_instructions(
    takehome_date, takehome_selected_step_source,
    task="Describe the user's chosen pattern in the wearable data",
) + (
    "\nFocus on that pattern, using numerical comparisons from the data. "
    "Distinguish raw points, rolling means, and missing observations; do not infer causes."
)
takehome_pattern_user_prompt = "Pattern to investigate:\n" + takehome_pattern_focus + "\n\nSelected data:\n" + takehome_compact_json
takehome_pattern_input_estimate = preview_openai_request(takehome_pattern_system_prompt, takehome_pattern_user_prompt)
TAKEHOME_PATTERN_PREVIEW = (takehome_pattern_context, takehome_pattern_system_prompt, takehome_pattern_user_prompt)

In [ ]:
#@title Provided - Part 4.6 - Send the reviewed pattern request
takehome_pattern_response = takehome_pattern_description = None
current_context = current_takehome_pattern_context()
require_reviewed_prompts(
    globals().get("TAKEHOME_PATTERN_PREVIEW"), current_context,
    globals().get("takehome_pattern_system_prompt"), globals().get("takehome_pattern_user_prompt"),
)
if openai_client is None:
    raise RuntimeError("Add your issued OPENAI_API_KEY in Colab Secrets and rerun Part 3.2 setup.")
takehome_pattern_response = openai_client.responses.create(
    model="gpt-5.6-luna", instructions=takehome_pattern_system_prompt, input=takehome_pattern_user_prompt,
    reasoning={"effort": "none"}, max_output_tokens=SUMMARY_MAX_OUTPUT_TOKENS, store=False,
)
takehome_pattern_description = takehome_pattern_response.output_text
display(Markdown(takehome_pattern_description))

## Troubleshooting

| Problem | What to do |
|---|---|
| Import or missing function | Rerun setup and the provided cells in order. |
| No device history yet | Use samples in class. Wear and sync your Fitbit before choosing a homework date; pairing does not create past sleep records. |
| Empty observations | Treat them as unavailable, not a measured zero. |
| Multiple step sources | Choose one exact package from the report. Never add separate source totals together. |
| No attributed step source | Choose another date/path; do not invent a shared source for unattributed points. |
| Missing Google Secrets | Add both matching values and enable notebook access. Class work can continue with samples. |
| Helper returns 400/401 | Stop retrying and reauthorize to replace both credentials. Use samples for class work while resolving access. |
| Helper returns 403 | Use the registered personal account and contact the course team privately. |
| Google returns 403 for a type | That scope may not have been granted; check authorization. |
| Google returns 401 | Reauthorize. You can explicitly switch to samples for class work. |
| Pagination repeats a token | Stop and report the status/type privately, without payloads or credentials. |
| Missing OpenAI key | Save your issued key as OPENAI_API_KEY in Colab Secrets, enable notebook access, and rerun Part 3.2 setup. Contact the course team privately if access still fails.  |
| Live Google/OpenAI error | Fix the reported problem. Class work can continue with samples while you resolve Fitbit access. No automatic fallback occurs. Clear failed-call outputs before submission. |
| Credential exposed | Revoke the course app in Google Account connections, delete both Secrets, then reauthorize. |

**Before submission:** clear raw live-record previews, the Part 3 live evidence
table, prompt previews from every request, and failed provider tracebacks.
Retain your three homework figures, reviewed AI summary and pattern description,
the classroom evidence check, and final two reflections. Remove identifying
names, account details, exact dates, and device metadata from retained titles and prose. Clock axes and numerical
results remain derived health information; course staff read these materials.
Never submit credentials or another person's records. Students without a Fitbit
submit the same analyses using the provided sample path.

## TODO: Summary and reflection

1. What did you learn about visualizing Fitbit records and checking a summary
   against them? Give one value from your run and one limit of its interpretation.
2. How did you use AI while completing this lab, and what did you verify yourself?

Write two or three sentences per response. Complete these now after the class
work, then update them after the homework if your conclusions change.

In [ ]:
#@title TODO - Summary and reflection
# TODO: Reflect on your own plots and evidence check.
topic_reflection = "TODO: one finding, its value, and a limit."
ai_use_reflection = "TODO: how you used AI and what you checked against your data."
display(Markdown(topic_reflection))
display(Markdown(ai_use_reflection))